# Module

In [3]:
import os
import glob
from pathlib import Path

import numpy as np
import pandas as pd

# Choose Data

In [7]:
# ======== CONFIGURATION ======== #

# To switch datasets, uncomment the appropriate lines below:

# Dataset: Brain
input_edgelist_path = "../data/brain/brain.txt"
dataset_name = "brain"

# Dataset: School
#input_edgelist_path = "../data/school/school.txt"
#dataset_name = "school"

# Dataset: Stock
#input_edgelist_path = "../data/stock/stock.txt"
#dataset_name = "stock"

# Dataset: Synthetic Experiment 1
#input_edgelist_path = "../data/synthetic_exp2.1_n200/synthetic_exp2.1_n200.txt"
#dataset_name = "synthetic_exp2.1"

# Dataset: Synthetic Experiment 2
#input_edgelist_path = "../data/synthetic_exp2.2_n200/synthetic_exp2.2_n200.txt"
#dataset_name = "synthetic_exp2.2"

# ======== OUTPUT SETUP ======== #
output_dir = f"./processed_data/{dataset_name}"
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Feature dimensions
node_feat_dim = 1
edge_feat_dim = 1

## SET THIS TO 1 FOR small datasets (synthetic and school)
if (dataset_name == "school") | (dataset_name == "synthetic_exp2.1") | (dataset_name == "synthetic_exp2.2"):
    random = 1

# Format

In [14]:
# ======== LOAD AND AUGMENT EDGES ======== #
df = pd.read_csv(input_edgelist_path, sep=' ', header=None, names=['u', 'i', 'ts'])
df_rev = pd.DataFrame({"u": df["i"], "i": df["u"], "ts": df["ts"]})

# Combine and drop exact duplicates
df_full = pd.concat([df, df_rev])
df_full.drop_duplicates(inplace=True)  # <- ADD THIS LINE
df_full = df_full.sort_values('ts').reset_index(drop=True)

# ======== ADD LABEL AND EDGE INDEX ======== #
df_full['label'] = 1.0
df_full['idx'] = df_full.index

# ======== GENERATE FEATURES ======== #
num_nodes = max(df_full['u'].max(), df_full['i'].max())
#node_feats = np.zeros((num_nodes + 1, node_feat_dim))  # node 0 is dummy
#edge_feats = np.zeros((df_full.shape[0] + 1, edge_feat_dim))  # edge 0 is dummy

# Node features
if random == 1:
    node_feats = np.random.randn(num_nodes + 1, node_feat_dim)  # random 1D feature
else:
    node_feats = np.zeros((num_nodes + 1, node_feat_dim))  # all-zero features

# Edge features
if random == 1:
    edge_feats = np.random.randn(df_full.shape[0]  + 1, edge_feat_dim)  # random 1D feature
else:
    edge_feats  = np.zeros((df_full.shape[0] + 1, edge_feat_dim))


# ======== SAVE FILES ======== #
df_full.to_csv(f"{output_dir}/ml_{dataset_name}.csv", index=False)
np.save(f"{output_dir}/ml_{dataset_name}.npy", edge_feats)
np.save(f"{output_dir}/ml_{dataset_name}_node.npy", node_feats)

# ======== LOG ======== #
print(f"Processed dataset saved to: {output_dir}")
print(f"Number of unique nodes: {len(set(df_full['u']).union(df_full['i']))}")
print(f"Number of edges (original + reverse, no exact duplicates): {df_full.shape[0]}")

Processed dataset saved to: ./processed_data/school
Number of unique nodes: 327
Number of edges (original + reverse, no exact duplicates): 22488
